In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

BASE_DIR = "/content/drive/MyDrive/PriceOptima_M2"

folders = [
    f"{BASE_DIR}/data/raw",
    f"{BASE_DIR}/data/processed",
    f"{BASE_DIR}/data/daily_ingest"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Folder structure created at:", BASE_DIR)

✅ Folder structure created at: /content/drive/MyDrive/PriceOptima_M2


In [3]:
from google.colab import files
uploaded = files.upload()

Saving inventory_data.csv to inventory_data.csv


In [4]:
from google.colab import files
uploaded = files.upload()

Saving sales_data.csv to sales_data.csv


In [5]:
import shutil, glob

raw_path = f"{BASE_DIR}/data/raw"
for f in glob.glob("/content/*.csv"):
    shutil.move(f, os.path.join(raw_path, os.path.basename(f)))

print("✅ Files moved to:", raw_path)

✅ Files moved to: /content/drive/MyDrive/PriceOptima_M2/data/raw


In [6]:
import os
raw_path = f"{BASE_DIR}/data/raw"
print("Raw files:", os.listdir(raw_path))

Raw files: ['inventory_data.csv', 'sales_data.csv']


In [7]:
ingest_code = r'''
import os
import sys
import pandas as pd
from datetime import datetime

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
DAILY_DIR = os.path.join(BASE_DIR, "data", "daily_ingest")

SALES_FILE = "sales_data.csv"
INV_FILE = "inventory_data.csv"

# Change date here if you want fixed run date (ex: 2026-02-19)
RUN_DATE = None  # None = today's date automatically

# Required columns (edit based on your dataset)
REQUIRED_SALES_COLS = ["date", "product_id", "selling_price", "units_sold"]
REQUIRED_INV_COLS   = ["date", "product_id", "inventory_level"]

def log(msg):
    print(msg)

def ensure_folders():
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(DAILY_DIR, exist_ok=True)

def get_run_date():
    if RUN_DATE:
        return RUN_DATE
    return datetime.now().strftime("%Y-%m-%d")

def validate_columns(df, required_cols, file_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"❌ Missing required columns in {file_name}: {missing}")

def clean_df(df):
    # Remove duplicates
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)

    # Strip column names
    df.columns = [c.strip() for c in df.columns]

    # Handle missing values (basic strategy)
    # - For numeric: fill with median
    # - For object/date: fill with mode (or 'Unknown' if mode missing)
    for col in df.columns:
        if df[col].dtype.kind in "biufc":  # numeric
            if df[col].isna().any():
                df[col] = df[col].fillna(df[col].median())
        else:
            if df[col].isna().any():
                mode = df[col].mode()
                df[col] = df[col].fillna(mode.iloc[0] if len(mode) > 0 else "Unknown")

    return df, (before, after)

def parse_dates_if_present(df):
    # If a column named 'date' exists, attempt conversion
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

def process_file(file_name, required_cols, output_name):
    in_path = os.path.join(RAW_DIR, file_name)

    if not os.path.exists(in_path):
        log(f"⚠️ {file_name} not found in raw/. Skipping.")
        return None

    log(f"📥 Loading: {file_name}")
    df = pd.read_csv(in_path)
    log(f"✅ {file_name} loaded successfully. Rows: {len(df)}")

    # Standardize columns early
    df.columns = [c.strip() for c in df.columns]

    validate_columns(df, required_cols, file_name)

    df = parse_dates_if_present(df)
    df, (before, after) = clean_df(df)

    log(f"🧹 Cleaning completed for {file_name} | Duplicates removed: {before - after}")

    # Save into processed/
    out_processed = os.path.join(PROCESSED_DIR, output_name)
    df.to_csv(out_processed, index=False)
    log(f"💾 Saved processed file: {out_processed}")

    return df

def main():
    ensure_folders()
    run_date = get_run_date()

    # Daily folder per run date
    daily_run_dir = os.path.join(DAILY_DIR, run_date)
    os.makedirs(daily_run_dir, exist_ok=True)

    log("🚀 Starting ingestion...")
    log(f"📅 Run Date Folder: {daily_run_dir}")

    # Sales
    sales_df = process_file(SALES_FILE, REQUIRED_SALES_COLS, "sales_cleaned.csv")
    if sales_df is not None:
        sales_daily_path = os.path.join(daily_run_dir, "sales_cleaned.csv")
        sales_df.to_csv(sales_daily_path, index=False)
        log(f"📦 Daily sales output saved: {sales_daily_path}")

    # Inventory (optional)
    inv_df = process_file(INV_FILE, REQUIRED_INV_COLS, "inventory_cleaned.csv")
    if inv_df is not None:
        inv_daily_path = os.path.join(daily_run_dir, "inventory_cleaned.csv")
        inv_df.to_csv(inv_daily_path, index=False)
        log(f"📦 Daily inventory output saved: {inv_daily_path}")

    log("✅ Ingestion completed successfully.")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print("❌ ERROR:", str(e))
        sys.exit(1)
'''

script_path = "/content/drive/MyDrive/PriceOptima_M2/ingest.py"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(ingest_code)

print("✅ ingest.py created at:", script_path)

✅ ingest.py created at: /content/drive/MyDrive/PriceOptima_M2/ingest.py


In [8]:
import re

script_path = "/content/drive/MyDrive/PriceOptima_M2/ingest.py"
with open(script_path, "r", encoding="utf-8") as f:
    txt = f.read()

txt = re.sub(r"RUN_DATE = None.*", "RUN_DATE = '2026-02-19'  # fixed run date for submission", txt)

with open(script_path, "w", encoding="utf-8") as f:
    f.write(txt)

print("✅ RUN_DATE set to 2026-02-19")

✅ RUN_DATE set to 2026-02-19


In [9]:
!python /content/drive/MyDrive/PriceOptima_M2/ingest.py

🚀 Starting ingestion...
📅 Run Date Folder: /content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19
📥 Loading: sales_data.csv
✅ sales_data.csv loaded successfully. Rows: 73100
❌ ERROR: ❌ Missing required columns in sales_data.csv: ['date', 'product_id', 'selling_price', 'units_sold']


In [10]:
import os
print(os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/processed"))

[]


In [11]:
import os
print(os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest"))
print(os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19"))

['2026-02-23', '2026-02-19', '2026-02-25']
[]


In [12]:
import os

path = "/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19"

print("Files in folder:")
print(os.listdir(path))

Files in folder:
[]


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os

path = "/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest"

print(os.listdir(path))

['2026-02-23', '2026-02-19', '2026-02-25']


In [16]:
path = "/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19"

print(os.listdir(path))

[]


In [17]:
import os

raw_path = "/content/drive/MyDrive/PriceOptima_M2/data/raw"
print("Raw folder files:", os.listdir(raw_path))

Raw folder files: ['inventory_data.csv', 'sales_data.csv']


In [18]:
!python /content/drive/MyDrive/PriceOptima_M2/ingest.py

🚀 Starting ingestion...
📅 Run Date Folder: /content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19
📥 Loading: sales_data.csv
✅ sales_data.csv loaded successfully. Rows: 73100
❌ ERROR: ❌ Missing required columns in sales_data.csv: ['date', 'product_id', 'selling_price', 'units_sold']


In [19]:
Starting ingestion...
Loading file: sales_data.csv
Saved to processed: ...
Saved to daily ingest: ...
Loading file: inventory_data.csv
Saved to processed: ...
Saved to daily ingest: ...
Ingestion completed successfully.

SyntaxError: invalid syntax (932416140.py, line 1)

In [20]:
!python /content/drive/MyDrive/PriceOptima_M2/ingest.py

🚀 Starting ingestion...
📅 Run Date Folder: /content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19
📥 Loading: sales_data.csv
✅ sales_data.csv loaded successfully. Rows: 73100
❌ ERROR: ❌ Missing required columns in sales_data.csv: ['date', 'product_id', 'selling_price', 'units_sold']


In [21]:
import os

print("Processed:", os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/processed"))
print("Daily:", os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19"))

Processed: []
Daily: []


In [22]:
code = '''
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/PriceOptima_M2"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
DAILY_DIR = os.path.join(BASE_DIR, "data", "daily_ingest")
RUN_DATE = "2026-02-19"

def clean_dataframe(df):
    df.columns = df.columns.str.strip()
    df = df.drop_duplicates()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("Unknown")
        else:
            df[col] = df[col].fillna(0)

    return df

def process_file(input_name, output_name):
    input_path = os.path.join(RAW_DIR, input_name)

    if not os.path.exists(input_path):
        print(f"File not found: {input_path}")
        return

    print(f"Loading file: {input_name}")
    df = pd.read_csv(input_path)
    print("Columns found:", list(df.columns))

    df = clean_dataframe(df)

    processed_path = os.path.join(PROCESSED_DIR, output_name)
    df.to_csv(processed_path, index=False)
    print(f"Saved to processed: {processed_path}")

    daily_folder = os.path.join(DAILY_DIR, RUN_DATE)
    os.makedirs(daily_folder, exist_ok=True)

    daily_path = os.path.join(daily_folder, output_name)
    df.to_csv(daily_path, index=False)
    print(f"Saved to daily ingest: {daily_path}")

def main():
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(DAILY_DIR, exist_ok=True)

    print("Starting ingestion...")

    process_file("sales_data.csv", "sales_cleaned.csv")
    process_file("inventory_data.csv", "inventory_cleaned.csv")

    print("Ingestion completed successfully.")

if __name__ == "__main__":
    main()
'''

with open("/content/drive/MyDrive/PriceOptima_M2/ingest.py", "w") as f:
    f.write(code)

print("ingest.py replaced successfully")

ingest.py replaced successfully


In [23]:
!python /content/drive/MyDrive/PriceOptima_M2/ingest.py

Starting ingestion...
Loading file: sales_data.csv
Columns found: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']
Saved to processed: /content/drive/MyDrive/PriceOptima_M2/data/processed/sales_cleaned.csv
Saved to daily ingest: /content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19/sales_cleaned.csv
Loading file: inventory_data.csv
Columns found: ['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']
Saved to processed: /content/drive/MyDrive/PriceOptima_M2/data/processed/inventory_cleaned.csv
Saved to daily ingest: /content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19/inventory_cleaned.csv
Ingestion completed successf

In [24]:
import os

print("Processed:", os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/processed"))
print("Daily:", os.listdir("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19"))

Processed: ['sales_cleaned.csv', 'inventory_cleaned.csv']
Daily: ['sales_cleaned.csv', 'inventory_cleaned.csv']


In [25]:
from google.colab import files

files.download("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19/sales_cleaned.csv")
files.download("/content/drive/MyDrive/PriceOptima_M2/data/daily_ingest/2026-02-19/inventory_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>